# KoboMart: Bronze Layer Ingestion (Auto Loader)

## **Dimension** Processing


In [0]:
# Import Required Libraries
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, IntegerType, DoubleType, DateType, TimestampType, StructField, FloatType
from  delta.tables import DeltaTable

In [0]:
%run /Workspace/Kobomart/kobomart_data_engineering_project/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
# Set Up Kobomart Catalog and Table Widgets
dbutils.widgets.text('catalog', 'kobomart', 'catalog')
dbutils.widgets.text('customer_source','customers', 'customer_source')
dbutils.widgets.text('date_source','date', 'date_source')
dbutils.widgets.text('order_item_source','order_items', 'order_item_source')
dbutils.widgets.text('payment_source','payments', 'payments_source')
dbutils.widgets.text('product_source','products_source', 'product_source')
dbutils.widgets.text('return_source','returns', 'return_source')
dbutils.widgets.text('seller_source','sellers_source', 'seller_source')
dbutils.widgets.text('support_ticket_source', 'support_tickets', 'support_ticket_source')

In [0]:
# Environment Setup: S3 Paths and Streaming Checkpoint
base_path       = "s3://kobomart"
landing_path    = f"{base_path}/landing"
checkpoint_path = f"{base_path}/_checkpoints/bronze"

## CUSTOMER

In [0]:

# Retrieve and Print Catalog & Brands Source Widget Values
catalog = dbutils.widgets.get('catalog')
customer_source = dbutils.widgets.get('customer_source')

print(catalog, customer_source)


In [0]:
# Configure Landing and Checkpoint Paths for Customers
source_path = f"{landing_path}/customers/"
checkpoint  = f"{checkpoint_path}/customers"

In [0]:
## Define Customer Schema for Bronze Ingestion
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("birth_date", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("marketing_opt_in", StringType(), True),
    StructField("customer_segment", StringType(), True),
 ])

In [0]:
# read data into dataframe
df = (spark.read\
    .format('csv')
    .schema(customers_schema)
    .option('header', True)
    .load(source_path)
    .withColumn("read_timestamp", F.current_timestamp()) ## add metadata column
    .select("*", "_metadata.file_name", "_metadata.file_path") ## metadata columns
)

In [0]:
# display(df.limit(5))

In [0]:
# Ingest Raw Data into Bronze Layer (Autoloader)
source_path = f"s3://kobomart/landing/{customer_source}/"
checkpoint  = f"{checkpoint_path}/customers"

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .option("pathGlobFilter", "*.csv")
    .schema(customers_schema)
    .load(source_path)
    .withColumn("ingest_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
 .writeStream
    .outputMode("append")
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(f"{catalog}.{bronze_schema}.{customer_source}")
    .awaitTermination())

## DATE

In [0]:
# Retrieve and Print Catalog & Brands Source Widget Values
catalog = dbutils.widgets.get('catalog')
date_source = dbutils.widgets.get('date_source')

print(catalog, date_source)

In [0]:
# Configure Landing and Checkpoint Paths for Customers
source_path = f"{landing_path}/date/"
checkpoint  = f"{checkpoint_path}/date"

In [0]:
## Define Customer Schema for Bronze Ingestion
date_schema = StructType([
    StructField('full_date', StringType(), True),
    StructField('date_key', StringType(), True),
    StructField('year', StringType(), True),
    StructField('quater', StringType(), True),
    StructField('month', StringType(), True),
    StructField('month_name', StringType(), True),
    StructField('year_month', StringType(), True),
    StructField('week_of_year', StringType(), True),
    StructField('day_of_month', StringType(), True),
    StructField('day_of_week', StringType(), True),
    StructField('day_name', StringType(), True),
    StructField('is_weekend', StringType(), True),
    StructField('is_month_end', StringType(), True),
    StructField('is_quarter_end', StringType(), True),
    StructField('is_year_end', StringType(), True)
])

In [0]:
# read data into dataframe
df = (spark.read\
      .format('csv')
      .schema(date_schema)
      .option('header', True)
      .load(source_path)
      .withColumn("read_timestamp", F.current_timestamp()) ## add metadata column
      .select("*", "_metadata.file_name", "_metadata.file_path") ## metadata columns
)

In [0]:
#display(df.limit(5))

In [0]:
# Ingest Raw Data into Bronze Layer (Autoloader)
source_path = f"s3://kobomart/landing/{date_source}/"
checkpoint  = f"{checkpoint_path}/date"

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .option("pathGlobFilter", "*.csv")
    .schema(date_schema)
    .load(source_path)
    .withColumn("ingest_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
 .writeStream
    .outputMode("append")
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(f"{catalog}.{bronze_schema}.{date_source}")
    .awaitTermination())